# Kaggle Train With Fast Git Sync

Notebook này dùng GPU Kaggle để train Tiny YOLO from scratch. Nếu repo đã có trong `/kaggle/working/XLA` thì chỉ đồng bộ code mới từ GitHub; nếu chưa có thì clone lần đầu.

In [ ]:
REPO_URL = "https://github.com/huyvanzzz/XLA.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working/XLA"

# Sua duong dan nay theo Kaggle Dataset cua ban. Auto-detect chi la fallback neu path nay sai.
KAGGLE_PUBLIC_DIR = "/kaggle/input/xla-object-detection/public"

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/kaggle/working")
work_path = Path(WORK_DIR)

if (work_path / ".git").exists():
    print("Repo exists, syncing latest code...")
    subprocess.run(["git", "-C", WORK_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    if work_path.exists():
        shutil.rmtree(work_path)
    print("Repo not found, cloning...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK_DIR], check=True)

os.chdir(WORK_DIR)
print("Using repo at", WORK_DIR)
!git log --oneline -1

In [ ]:
# Kaggle có thể dùng PyTorch quá mới không hỗ trợ Tesla P100 (sm_60).
# Bản này tương thích P100 và vẫn chạy tốt trên GPU Kaggle phổ biến.
!python -m pip uninstall -y -q torch torchvision torchaudio
!python -m pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 torch==2.4.1+cu121
!python -m pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import shutil

def find_public_dir() -> Path:
    candidates = []
    if KAGGLE_PUBLIC_DIR:
        candidates.append(Path(KAGGLE_PUBLIC_DIR))
    candidates.extend(path.parent for path in Path("/kaggle/input").rglob("classes.json"))
    for candidate in candidates:
        if (
            (candidate / "classes.json").exists()
            and (candidate / "annotations" / "train.json").exists()
            and (candidate / "annotations" / "val.json").exists()
            and (candidate / "train" / "images").exists()
            and (candidate / "val" / "images").exists()
        ):
            return candidate
    searched = ", ".join(str(path) for path in candidates[:20])
    raise FileNotFoundError(f"Cannot auto-detect public dataset. Checked: {searched}")

src_public = find_public_dir()
dst_public = Path(WORK_DIR) / "public"

if src_public.exists():
    if dst_public.exists():
        shutil.rmtree(dst_public)
    shutil.copytree(src_public, dst_public)
    print("Copied dataset from", src_public)
elif dst_public.exists():
    print("Using existing public/ inside repo")
else:
    raise FileNotFoundError(f"Cannot find dataset at {src_public}. Upload public/ as a Kaggle Dataset.")

print("classes.json:", dst_public / "classes.json")

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import copy
import yaml

FINETUNE_MODE = "strong"  # "safe" keeps the previous conservative run; "strong" is the final higher-upside try.

base_config = Path("./configs/default.yaml")
safe_config = Path("./configs/finetune_from_best.yaml")
strong_config = Path("./configs/finetune_from_best_strong.yaml")
with base_config.open("r", encoding="utf-8") as f:
    base = yaml.safe_load(f)


def apply_common(cfg):
    cfg["validation_metric"].update({
        "tta_hflip": True,
        "tta_fusion": "wbf",
        "tta_iou_threshold": 0.55,
        "tta_start_epoch": 1,
        "tta_every": 1,
        "distribution_quality_power": 0.35,
        "conf_threshold": 0.01,
        "nms_threshold": 0.5,
    })
    cfg["inference"].update({
        "tta_hflip": True,
        "tta_fusion": "wbf",
        "tta_iou_threshold": 0.55,
        "distribution_quality_power": 0.35,
        "conf_threshold": 0.01,
        "nms_threshold": 0.5,
    })
    return cfg


safe = apply_common(copy.deepcopy(base))
safe.update({
    "epochs": 18,
    "lr": 2.0e-5,
    "lr_final_factor": 0.20,
    "fine_tune_lr_factor": 1.0,
    "backbone_lr_mult": 0.05,
    "weight_decay": 0.008,
    "warmup_epochs": 1,
    "freeze_backbone_epochs": 2,
    "early_stopping_patience": 8,
})
safe["augmentation"].update({
    "mosaic_prob": 0.0,
    "close_mosaic_epoch": 1,
    "random_crop_prob": 0.0,
    "random_scale_prob": 0.0,
    "random_erasing_prob": 0.0,
    "color_jitter_prob": 0.10,
    "color_jitter_min": 0.9,
    "color_jitter_max": 1.1,
    "close_strong_aug_epoch": 1,
})

# Strong-long keeps checkpoint weights, trains longer, and gives chair/car more pixels plus mild rebalancing.
strong = apply_common(copy.deepcopy(safe))
strong.update({
    "image_size": 544,
    "batch_size": 12,
    "val_batch_size": 24,
    "epochs": 36,
    "lr": 1.2e-5,
    "lr_final_factor": 0.18,
    "warmup_epochs": 2,
    "freeze_backbone_epochs": 1,
    "early_stopping_patience": 12,
})
strong["class_weights"]["overrides"] = {"chair": 1.55, "car": 1.12}
strong["balanced_sampling"].update({"enabled": True, "power": 0.35, "empty_weight": 0.20})
strong["augmentation"].update({
    "random_scale_prob": 0.08,
    "min_scale": 0.95,
    "max_scale": 1.05,
    "color_jitter_prob": 0.12,
    "color_jitter_min": 0.92,
    "color_jitter_max": 1.08,
    "close_strong_aug_epoch": 28,
})

for cfg, path in [(safe, safe_config), (strong, strong_config)]:
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    print("Wrote", path)

active_config = strong_config if FINETUNE_MODE == "strong" else safe_config
active_dir = Path("./models_finetune_strong" if FINETUNE_MODE == "strong" else "./models_finetune")
print("FINETUNE_MODE=", FINETUNE_MODE)
print("ACTIVE_CONFIG=", active_config)
print("ACTIVE_DIR=", active_dir)


In [ ]:
from pathlib import Path

resume_checkpoint = Path("./models/best.pth")
if not resume_checkpoint.exists():
    raise FileNotFoundError("Missing ./models/best.pth. Upload/copy the current best checkpoint before fine-tuning.")

!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir {active_dir}/ \
  --resume_checkpoint ./models/best.pth \
  --config {active_config} \
  --classes ./public/classes.json


In [ ]:
!python predict.py \
  --image_dir ./public/val/images \
  --output ./val_predictions_{FINETUNE_MODE}.json \
  --checkpoint {active_dir}/best.pth \
  --config {active_config} \
  --classes ./public/classes.json \
  --batch_size 16 \
  --distribution_quality_power 0.35

!python public/tools/evaluate_predictions.py \
  --ground_truth ./public/annotations/val.json \
  --predictions ./val_predictions_{FINETUNE_MODE}.json \
  --output ./val_score_{FINETUNE_MODE}.json

!cat ./val_score_{FINETUNE_MODE}.json


In [ ]:
from pathlib import Path
import shutil

artifact_dir = Path("/kaggle/working/artifacts")
artifact_dir.mkdir(exist_ok=True)
for src_dir_name in ["models", "models_finetune", "models_finetune_strong"]:
    src_dir = Path(src_dir_name)
    if not src_dir.exists():
        continue
    dst_dir = artifact_dir / src_dir_name
    dst_dir.mkdir(exist_ok=True)
    for name in ["best.pth", "last.pth", "history.jsonl"]:
        src = src_dir / name
        if src.exists():
            shutil.copy2(src, dst_dir / name)
for name in [f"val_predictions_{FINETUNE_MODE}.json", f"val_score_{FINETUNE_MODE}.json"]:
    src = Path(name)
    if src.exists():
        shutil.copy2(src, artifact_dir / name)
for config_name in ["finetune_from_best.yaml", "finetune_from_best_strong.yaml"]:
    config_src = Path("./configs") / config_name
    if config_src.exists():
        shutil.copy2(config_src, artifact_dir / config_name)
print("Artifacts:", sorted(p.name for p in artifact_dir.iterdir()))
